# MiniProject 2 — Part 2 Visualization

This notebook regenerates the monthly commit time series and plots from `asaucer_project_summary.csv`.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

NETID = 'asaucer'
summary = pd.read_csv(f'{NETID}_project_summary.csv')
summary['time'] = pd.to_numeric(summary['time'], errors='coerce')
summary = summary.dropna(subset=['time'])

In [2]:
def monthly_commits(project):
    dates = pd.to_datetime(project['time'], unit='s', utc=True).dt.tz_localize(None)
    counts = dates.dt.to_period('M').value_counts().sort_index()
    months = pd.period_range(counts.index.min(), counts.index.max(), freq='M')
    return pd.DataFrame({'Month': months.astype(str), '#Commits': counts.reindex(months, fill_value=0).astype(int).to_numpy()})

for project, records in summary.groupby('project_wocid', sort=True):
    monthly = monthly_commits(records)
    monthly.to_csv(f'{NETID}_commits_timeseries_{project}.csv', sep=';', index=False)
    plt.figure(figsize=(12, 6))
    plt.plot(monthly['Month'], monthly['#Commits'], marker='o', linewidth=1.5, markersize=3)
    plt.xlabel('Time (YYYY-MM)')
    plt.ylabel('Number of Commits')
    plt.title(f'Commit Activity: {project}')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f'{NETID}_timeseries_{project}.png', dpi=300)
    plt.close()

print('Generated monthly CSVs and 300-DPI plots for', summary['project_wocid'].nunique(), 'projects.')

Generated monthly CSVs and 300-DPI plots for 10 projects.
